# Lab 3: Amazon Bedrock Guardrails

This lab walks through creating an Amazon Bedrock Guardrail, testing it directly with the
`ApplyGuardrail` API, and wiring it into a Strands agent for the NovaPay AI Support Agent.

**Prerequisites**

- AWS CLI credentials configured with `bedrock:CreateGuardrail`, `bedrock:ApplyGuardrail`,
  `bedrock:DeleteGuardrail`, and `bedrock:InvokeModel` permissions
- Model access enabled for the Claude Haiku model used below, in your target region
- Packages installed from `requirements.txt` (`strands-agents`, `strands-agents-tools`, `boto3`)

## 1. Setup

In [ ]:
import json
import time
import boto3
from botocore.config import Config as BotocoreConfig
from IPython.display import display, HTML

from strands import Agent, tool
from strands.models import BedrockModel

# AWS clients — set REGION to match where your account has Bedrock and guardrails enabled.
REGION = "us-east-1"
bedrock_client = boto3.client("bedrock", region_name=REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

print("✅ Setup complete")

## 2. What Are Bedrock Guardrails?

Guardrails are server-side content filters that sit between your app and the LLM:

```
User Input → [GUARDRAIL INPUT CHECK] → LLM → [GUARDRAIL OUTPUT CHECK] → Response
                    ↓                                    ↓
              Block/Modify                        Block/Anonymize
```

### Guardrail Components

| Component | Purpose | NovaPay Use Case |
|---|---|---|
| Denied Topics | Block entire categories | No investment advice |
| Content Filters | Block harmful content | No hate/violence |
| PII Filters | Detect/mask personal data | Mask SSN, anonymize card numbers |
| Word Filters | Block specific words | Block competitor names |
| Contextual Grounding | Prevent hallucination | Verify claims against source |

### Why Guardrails Matter for Fintech

- **Regulatory compliance**: Can't give investment advice without a license
- **Data protection**: PII must never leak in logs or responses
- **Brand safety**: Agent must stay on-topic (payments, not politics)

## 3. Create a Bedrock Guardrail via API

We'll create a guardrail with:

1. **Denied topic**: Investment advice (NovaPay is payments, not a brokerage)
2. **PII filters**: Block SSN display, anonymize card numbers

In [ ]:
# Create the NovaPay guardrail
GUARDRAIL_NAME = f"novapay-lab-guardrail-{int(time.time())}"

try:
    create_response = bedrock_client.create_guardrail(
        name=GUARDRAIL_NAME,
        description="NovaPay agent guardrail: blocks investment advice, protects PII",
        topicPolicyConfig={
            "topicsConfig": [
                {
                    "name": "Investment Advice",
                    "definition": "Any recommendations about buying, selling, or holding stocks, bonds, cryptocurrency, or other financial instruments. Includes portfolio allocation suggestions and market predictions.",
                    "examples": [
                        "Should I buy Bitcoin?",
                        "What stocks should I invest in?",
                        "Is now a good time to invest in crypto?",
                        "Recommend a portfolio allocation for retirement"
                    ],
                    "type": "DENY"
                }
            ]
        },
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
                {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "ANONYMIZE"},
                {"type": "EMAIL", "action": "ANONYMIZE"}
            ]
        },
        contentPolicyConfig={
            "filtersConfig": [
                {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"}
            ]
        },
        blockedInputMessaging="I\'m sorry, I can only help with NovaPay payment and account questions. I cannot provide investment advice.",
        blockedOutputsMessaging="I\'m sorry, I cannot share that information due to our data protection policies."
    )

    GUARDRAIL_ID = create_response["guardrailId"]
    GUARDRAIL_VERSION = "DRAFT"  # Use DRAFT for testing

    display(HTML(f'<h3 style="color: #2ecc71;">✅ Guardrail created: {GUARDRAIL_ID}</h3>'))
    print(f"Name: {GUARDRAIL_NAME}")
    print(f"ID: {GUARDRAIL_ID}")
    print(f"Version: {GUARDRAIL_VERSION}")

except Exception as e:
    print(f"❌ Error creating guardrail: {e}")
    print("\nIf you get an access error, ensure your IAM role has bedrock:CreateGuardrail permission.")
    # Set fallback values for the rest of the lab
    GUARDRAIL_ID = None
    GUARDRAIL_VERSION = None

## 4. Test Guardrail Directly with ApplyGuardrail API

Before wiring to an agent, test the guardrail in isolation.
The `ApplyGuardrail` API lets you test inputs/outputs without calling an LLM.

In [ ]:
def test_guardrail(text: str, source: str = "INPUT"):
    """Test a text against our guardrail."""
    if not GUARDRAIL_ID:
        print("⚠️  No guardrail ID available. Skipping test.")
        return None

    try:
        response = bedrock_runtime.apply_guardrail(
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion=GUARDRAIL_VERSION,
            source=source,
            content=[{"text": {"text": text}}]
        )

        action = response["action"]
        outputs = response.get("outputs", [])
        assessments = response.get("assessments", [])

        preview = text[:80] + "..." if len(text) > 80 else text
        print(f'📝 Input: "{preview}"')
        print(f"🎯 Action: {action}")

        if outputs:
            output_text = outputs[0].get("text", "N/A")
            print(f"📤 Output: {output_text}")

        if assessments:
            for assessment in assessments:
                if "topicPolicy" in assessment:
                    topics = assessment["topicPolicy"].get("topics", [])
                    for topic in topics:
                        print(f"   🚫 Denied topic: {topic['name']} (action: {topic['action']})")
                if "sensitiveInformationPolicy" in assessment:
                    pii = assessment["sensitiveInformationPolicy"].get("piiEntities", [])
                    for entity in pii:
                        print(f"   🔒 PII detected: {entity['type']} (action: {entity['action']})")

        return response
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ Test function defined")

In [ ]:
# Test 1: Safe input (should pass)
print("=" * 60)
print("TEST 1: Safe input – balance inquiry")
print("=" * 60)
test_guardrail("What is my account balance?")

print("\n")

# Test 2: Denied topic (should block)
print("=" * 60)
print("TEST 2: Denied topic – investment advice")
print("=" * 60)
test_guardrail("Should I invest in Bitcoin? What stocks do you recommend for my portfolio?")

print("\n")

# Test 3: PII in output (should anonymize)
print("=" * 60)
print("TEST 3: PII – credit card number in output")
print("=" * 60)
test_guardrail("The customer\'s card number is 4532-1234-5678-9012 and their email is amara@example.com", source="OUTPUT")

## 5. Wire Guardrail to a Strands Agent

Strands' `BedrockModel` accepts `guardrail_id` and `guardrail_version` parameters.
When set, every request passes through the guardrail automatically.

In [ ]:
# Create a model with guardrail enforcement
if GUARDRAIL_ID:
    guarded_model = BedrockModel(
        model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        guardrail_id=GUARDRAIL_ID,
        guardrail_version=GUARDRAIL_VERSION,
        boto_client_config=BotocoreConfig(
            retries={"max_attempts": 3},
            connect_timeout=5,
            read_timeout=60
        )
    )

    display(HTML('<h3 style="color: #2ecc71;">✅ Guarded model created</h3>'))
    print(f"Guardrail ID: {GUARDRAIL_ID}")
    print(f"Guardrail Version: {GUARDRAIL_VERSION}")
else:
    print("⚠️  Using unguarded model (no guardrail ID available)")
    guarded_model = BedrockModel(
        model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        boto_client_config=BotocoreConfig(
            retries={"max_attempts": 3},
            connect_timeout=5,
            read_timeout=60
        )
    )

In [ ]:
# NovaPay data for the agent
NOVAPAY_CUSTOMERS = {
    "CUST-1001": {"name": "Amara Okafor", "balance": 15420.50, "currency": "NGN", "account_type": "premium"},
    "CUST-1002": {"name": "Kwame Mensah", "balance": 3200.00, "currency": "GHS", "account_type": "standard"}
}

NOVAPAY_TRANSACTIONS = {
    "TXN-101": {"customer_id": "CUST-1001", "amount": 2500.00, "currency": "NGN", "type": "transfer", "status": "completed", "timestamp": "2024-01-15T10:30:00Z", "recipient": "Chidi Nwankwo", "fraud_flags": []},
    "TXN-102": {"customer_id": "CUST-1001", "amount": 89000.00, "currency": "NGN", "type": "transfer", "status": "flagged", "timestamp": "2024-01-15T23:45:00Z", "recipient": "Unknown Merchant XYZ", "fraud_flags": ["unusual_amount", "odd_hour", "new_recipient"]},
    "TXN-103": {"customer_id": "CUST-1002", "amount": 150.00, "currency": "GHS", "type": "payment", "status": "completed", "timestamp": "2024-01-16T08:00:00Z", "recipient": "Accra Utilities", "fraud_flags": []}
}

@tool
def check_balance(customer_id: str) -> str:
    """Check account balance for a NovaPay customer.

    Args:
        customer_id: The customer ID (e.g., CUST-1001)
    """
    customer = NOVAPAY_CUSTOMERS.get(customer_id)
    if not customer:
        return json.dumps({"error": f"Customer {customer_id} not found"})
    return json.dumps({"customer_id": customer_id, "name": customer["name"], "balance": customer["balance"], "currency": customer["currency"]})

@tool
def get_transactions(transaction_id: str) -> str:
    """Get details of a NovaPay transaction.

    Args:
        transaction_id: The transaction ID (e.g., TXN-101)
    """
    txn = NOVAPAY_TRANSACTIONS.get(transaction_id)
    if not txn:
        return json.dumps({"error": f"Transaction {transaction_id} not found"})
    return json.dumps(txn)

# Create guarded agent
guarded_agent = Agent(
    model=guarded_model,
    tools=[check_balance, get_transactions],
    system_prompt="""You are NovaPay\'s AI Support Agent.
Help customers with account balances and transaction inquiries.
You handle PAYMENTS only. Never provide investment or financial planning advice."""
)

display(HTML('<h3 style="color: #2ecc71;">✅ Guarded NovaPay agent created</h3>'))

## 6. Test: Agent with Guardrail Blocks Denied Topics

In [ ]:
# Test: Normal query (should work)
print("=" * 60)
print("TEST: Normal query – should PASS through guardrail")
print("=" * 60)
try:
    response = guarded_agent("What is the balance for customer CUST-1001?")
    print(f"\n🤖 Agent: {response}")
except Exception as e:
    print(f"Response: {e}")

In [ ]:
# Test: Investment advice (should be BLOCKED by guardrail)
print("=" * 60)
print("TEST: Investment advice – should be BLOCKED by guardrail")
print("=" * 60)
try:
    response = guarded_agent("Should I invest my NovaPay balance in Bitcoin? What\'s the best crypto portfolio allocation?")
    print(f"\n🤖 Agent: {response}")
    print("\n💡 If the guardrail worked, you should see the blocked message above.")
except Exception as e:
    print(f"\n🚫 Blocked by guardrail: {e}")
    print("\n💡 The guardrail intercepted the request before it reached the LLM!")

## 7. Test: PII in Output is Anonymized

Even if the agent tries to include sensitive data in its response,
the guardrail's OUTPUT filter will catch and anonymize it.

In [ ]:
# Test PII anonymization on output side
print("=" * 60)
print("TEST: PII anonymization")
print("=" * 60)

# Test directly with ApplyGuardrail (OUTPUT direction)
if GUARDRAIL_ID:
    pii_test_text = "Customer Amara Okafor\'s card is 4532-1234-5678-9012 and SSN is 123-45-6789. Email: amara.okafor@example.com"
    print(f"\n📝 Original output: {pii_test_text}")
    print("\n🔒 After guardrail processing:")
    result = test_guardrail(pii_test_text, source="OUTPUT")
else:
    print("⚠️  Guardrail not available. In production, PII would be anonymized like:")
    print("   Original: \'Card is 4532-1234-5678-9012\'")
    print("   Filtered: \'Card is {CREDIT_DEBIT_CARD_NUMBER}\'")

## 8. Teardown — Delete the Guardrail

Always clean up resources after lab exercises to avoid costs.

In [ ]:
# Delete the guardrail
if GUARDRAIL_ID:
    try:
        bedrock_client.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
        display(HTML(f'<h3 style="color: #e74c3c;">🗑️ Guardrail {GUARDRAIL_ID} deleted</h3>'))
        print(f"Deleted: {GUARDRAIL_NAME}")
    except Exception as e:
        print(f"⚠️  Could not delete guardrail: {e}")
else:
    print("No guardrail to delete.")

## 9. Summary: Guardrail Architecture

```
┌─────────────────────────────────────────────┐
│              BEDROCK GUARDRAIL               │
│                                               │
│  INPUT FILTERS:        OUTPUT FILTERS:       │
│  ┌────────────────┐    ┌──────────────────┐  │
│  │ Denied Topics   │    │ PII Anonymize    │  │
│  │ Content Filter  │    │ Content Filter   │  │
│  │ Word Filter     │    │ Word Filter      │  │
│  └────────┬────────┘    └────────┬─────────┘  │
│           ▼                      ▼            │
│    [Block or Pass]        [Anonymize or Pass] │
└─────────────────────────────────────────────┘
```

### Key Takeaways

- Guardrails are **server-side** — they work regardless of client implementation
- **Input filters** block harmful/off-topic requests BEFORE they reach the LLM
- **Output filters** sanitize LLM responses BEFORE they reach the user
- In Strands, just pass `guardrail_id` and `guardrail_version` to `BedrockModel`

### 🧠 Knowledge Check

1. What's the difference between BLOCK and ANONYMIZE PII actions?
2. Why test with `ApplyGuardrail` before wiring to an agent?
3. Where does the guardrail sit in the request flow — client-side or server-side?
4. What happens if a tool's output contains PII — does the guardrail catch it?
5. For a fintech app, what additional denied topics would you configure?

### Answers

<details>
<summary>Click to reveal</summary>

1. **BLOCK** rejects the request/response entirely when the PII type is detected. **ANONYMIZE** replaces
   the detected PII with a placeholder (e.g., `{CREDIT_DEBIT_CARD_NUMBER}`) and lets the rest of the
   content through.
2. Testing with `ApplyGuardrail` directly lets you validate policy behavior without incurring the cost
   and latency of an LLM call, and without needing a fully wired agent.
3. Server-side — the guardrail runs on Bedrock's infrastructure between the request and the model (and
   between the model and the response), so it applies no matter what client calls it.
4. Yes — the guardrail's OUTPUT filter inspects whatever text ultimately gets returned to the user,
   including text that originated from a tool call, and will anonymize/block PII found there.
5. Examples: unlicensed lending/credit advice, tax advice, legal advice, and anything resembling
   guaranteed-return promises or account-takeover assistance.

</details>

### ➡️ Next Steps

In **Lab 4**, you'll learn:

- Orchestration patterns (sequential, orchestrator/specialist, agent-as-a-tool)
- When to use each pattern
- Decision frameworks for FDEs

*NovaPay AI Agent Training — Lab 3 Complete*